# Evidencias individuales de Reconocimientos (menciones de honor)

Construye `data/evidencias/menciones_honor/evidencias_menciones_honor.csv` (`MENCION_HONOR`).
Reglas en `README.md` y formato de atributos en `../ATRIBUTOS.md`. Requiere haber generado antes las evidencias de trayectoria.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "02_evidencias").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from evidencias.reconocimientos import construir_evidencias_reconocimientos

pd.set_option("display.max_colwidth", 200)

In [ ]:
evidencias, reporte, validacion = construir_evidencias_reconocimientos(guardar=True)
print(json.dumps(reporte, ensure_ascii=False, indent=2))

## Validación (todas las filas deben tener `n_fallas = 0`)

In [ ]:
validacion

## Ejemplos

In [ ]:
for _, r in evidencias.sample(8, random_state=7).iterrows():
    print(f"- {r.evidencia_id} | {r.texto}")
    print("  ", json.dumps(json.loads(r.atributos), ensure_ascii=False))

## Casos recibidos más de una vez

Registros de la fuente y la evidencia que generan: `n_veces` cuenta cuántas veces la recibió y `fechas` lista cada fecha en que la recibió.

In [ ]:
from evidencias.reconocimientos.fuentes import cargar_menciones, cargar_poblacion

attrs = evidencias["atributos"].map(json.loads)
varias = evidencias[attrs.map(lambda a: a["n_veces"] > 1)]
print(f"Evidencias recibidas más de una vez: {len(varias)}")

fuente = cargar_menciones(cargar_poblacion())
cols = ["NOMBREMENCION", "INSTITUCION", "PAIS", "FECHA", "TIPO"]
for _, r in varias.sample(5, random_state=3).iterrows():
    a = json.loads(r["atributos"])
    print()
    print("=" * 90)
    print("Registros en la fuente:")
    print(fuente[fuente["IDMENCIONHONOR"].isin(a["_origen"]["ids_mencion_honor"])][cols].sort_values("FECHA").to_string(index=False))
    print()
    print("Texto:", r["texto"])
    for k in ["nombre", "nombres_originales", "ediciones", "tipos", "n_veces", "fechas", "instituciones", "paises", "es_exterior"]:
        print(f"  {k}: {a[k]}")

## Registros no considerados (para revisión)

In [ ]:
pd.read_csv(PROJECT_ROOT / "data" / "evidencias" / "menciones_honor" / "registros_no_considerados.csv")["motivo"].value_counts()